In [1]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "plots").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate the XAIV project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
PLOT_DIR = PROJECT_ROOT / "plots" / "5.5.1-1_new_L0"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = PLOT_DIR / ".cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
except ImportError:
    set_matplotlib_formats = None

FORCE_TEX = True
CSV_PATH = PROJECT_ROOT / "results" / "vggnet16" / "all_k_vggnet16.csv"
PDF_OUTPUT_PATH = PLOT_DIR / "imagenet_percent_changed_cumulative.pdf"
DEFAULT_FIGSIZE = (7.2, 4.8)
BASE_FONT_SIZE = 16
AXIS_LABEL_SIZE = 18
AXIS_TITLE_SIZE = 18
TICK_LABEL_SIZE = 16
QUANTILE_LABEL_SIZE = 11
LINE_COLOR = "#111111"


def latex_ready():
    required_commands = ("latex", "dvipng")
    if any(shutil.which(command) is None for command in required_commands):
        return False

    if shutil.which("kpsewhich") is not None:
        latex_fmt = subprocess.run(
            ["kpsewhich", "latex.fmt"],
            capture_output=True,
            text=True,
            check=False,
        )
        if latex_fmt.returncode != 0 or not latex_fmt.stdout.strip():
            return False

    latex_source = "\n".join([
        r"\documentclass{article}",
        r"\begin{document}",
        r"lp",
        r"\end{document}",
    ])

    with tempfile.TemporaryDirectory() as tmpdir:
        tex_path = Path(tmpdir) / "matplotlib_tex_probe.tex"
        tex_path.write_text(latex_source, encoding="utf-8")
        result = subprocess.run(
            ["latex", "-interaction=nonstopmode", "--halt-on-error", tex_path.name],
            cwd=tmpdir,
            capture_output=True,
            text=True,
            check=False,
        )
        return result.returncode == 0 and (Path(tmpdir) / "matplotlib_tex_probe.dvi").exists()


def has_tex_package(package_name):
    if shutil.which("kpsewhich") is None:
        return False
    return subprocess.run(
        ["kpsewhich", f"{package_name}.sty"],
        capture_output=True,
        text=True,
        check=False,
    ).returncode == 0


def configure_plot_style(force_tex=FORCE_TEX):
    use_tex = force_tex and latex_ready()
    preamble = r"\usepackage{fontawesome5}" if use_tex and has_tex_package("fontawesome5") else ""

    if set_matplotlib_formats is not None:
        if use_tex and shutil.which("dvisvgm") is not None:
            set_matplotlib_formats("svg")
        else:
            set_matplotlib_formats("png")

    if force_tex and not use_tex:
        print("LaTeX was requested but no compatible renderer was found. Falling back to Matplotlib serif text.")

    mpl.rcParams.update({
        "text.usetex": use_tex,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "text.latex.preamble": preamble,
        "font.size": BASE_FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": AXIS_TITLE_SIZE,
        "xtick.labelsize": TICK_LABEL_SIZE,
        "ytick.labelsize": TICK_LABEL_SIZE,
        "axes.linewidth": 1.0,
        "grid.linewidth": 0.6,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    return use_tex


def ecdf(values):
    values = np.sort(np.asarray(values, dtype=float))
    if values.size == 0:
        return np.array([]), np.array([])
    cumulative = np.arange(1, values.size + 1) / values.size
    return values, cumulative


def add_quantile_guides(ax, values, color, linestyle, label_side):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return

    x_min = max(0.0, float(values.min()))
    x_pad = 2.0 if label_side == "right" else -2.0
    x_align = "left" if label_side == "right" else "right"
    quantile_specs = [
        (0.10, 0.015, "bottom"),
        (0.50, 0.015, "bottom"),
        (0.90, -0.015, "top"),
    ]

    for q, y_offset, va in quantile_specs:
        x_q = float(np.quantile(values, q))
        y_q = q
        ax.hlines(
            y=y_q,
            xmin=x_min,
            xmax=x_q,
            color=color,
            linestyle=linestyle,
            linewidth=1.0,
            alpha=0.22,
            zorder=1,
        )
        ax.vlines(
            x=x_q,
            ymin=0.0,
            ymax=y_q,
            color=color,
            linestyle=linestyle,
            linewidth=1.0,
            alpha=0.22,
            zorder=1,
        )
        ax.text(
            np.clip(x_q + x_pad, 0.5, 99.5),
            np.clip(y_q + y_offset, 0.02, 0.98),
            f"{x_q:.1f}%",
            fontsize=QUANTILE_LABEL_SIZE,
            ha=x_align,
            va=va,
            color=color,
            clip_on=False,
        )


def load_imagenet_percent_changed():
    csv_path = CSV_PATH.resolve()
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    for col in ["total", "k", "eps", "percent_changed"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["tag", "image", "model", "result"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    df = df.dropna(subset=["total", "k", "percent_changed"]).copy()
    df = df[df["tag"] == "fix_nonmask"].copy()
    df = df[df["total"].astype(int) == 150528].copy()
    df = df.sort_values(["k", "percent_changed", "image"]).reset_index(drop=True)

    if df.empty:
        raise ValueError("No ImageNet fix_nonmask rows remain after filtering.")

    return csv_path, df


def plot_imagenet_percent_changed():
    csv_path, df = load_imagenet_percent_changed()
    values = df["percent_changed"].astype(float).to_numpy()
    x_ecdf, y_ecdf = ecdf(values)

    fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
    ax.plot(
        x_ecdf,
        y_ecdf,
        color=LINE_COLOR,
        linestyle="-",
        linewidth=3.0,
        marker="",
        solid_capstyle="butt",
        dash_capstyle="butt",
    )
    add_quantile_guides(ax, values, color=LINE_COLOR, linestyle="-", label_side="right")

    ax.set_xlim(0.0, 100.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xticks(np.linspace(0.0, 100.0, 6))
    ax.set_yticks(np.linspace(0.0, 1.0, 6))
    ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)
    ax.grid(axis="y", linestyle="--", alpha=0.28)
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    fig.suptitle("ImageNet", fontsize=AXIS_TITLE_SIZE, y=0.955)
    fig.supxlabel("Percent changed in the object-only perturbation", y=0.05, fontsize=AXIS_LABEL_SIZE)
    fig.supylabel("Cumulative fraction of images", x=0.025, fontsize=AXIS_LABEL_SIZE)
    fig.subplots_adjust(top=0.82, bottom=0.18, left=0.11, right=0.98)
    fig.savefig(PDF_OUTPUT_PATH, bbox_inches="tight")

    print(f"Using CSV: {csv_path}")
    print(f"Rows plotted: {len(df)}")
    print(f"Unique k values: {sorted(df['k'].dropna().astype(int).unique().tolist())}")
    print(f"Result counts: {df['result'].value_counts().to_dict()}")
    print(f"Percent changed summary: min={values.min():.3f}, median={np.median(values):.3f}, max={values.max():.3f}")
    print(f"Saved: {PDF_OUTPUT_PATH}")

    if "agg" in mpl.get_backend().lower():
        plt.close(fig)
    else:
        plt.show()


configure_plot_style()
plot_imagenet_percent_changed()


LaTeX was requested but no compatible renderer was found. Falling back to Matplotlib serif text.


Using CSV: /Users/zd3504phd/Desktop/XAIV/results/vggnet16/all_k_vggnet16.csv
Rows plotted: 1086
Unique k values: [1568, 3136, 6272, 12544, 25088, 50176]
Result counts: {'unsat False': 486, 'sat True': 366, 'sat False': 201, 'timeout False': 33}
Percent changed summary: min=3.125, median=17.990, max=99.006
Saved: /Users/zd3504phd/Desktop/XAIV/plots/5.5.1-1_new_L0/imagenet_percent_changed_cumulative.pdf
